<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/00_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# pytidb クイックスタート (TODO タスク管理)

このノートブックはGoogle Colabで学ぶ **pytidb シリーズの第 0 回** です。pytidb で TiDB に接続し、テーブル作成から CRUD までの一連を 15分程度で体験します。

## 学習目標
- `TiDBClient.connect()` で TiDB Cloud Zero に接続する
- `TableModel` でテーブルを Python のクラスとして定義する
- `insert` / `query` / `update` / `delete` で基本操作を行う


## 1. 依存パッケージのインストール

pytidbをインストールします。pytidbはpipでインストールできます。

In [ ]:
!pip install -q pytidb


## 2. TiDB Cloud Zero でクラスタを作成する

[TiDB Cloud Zero](https://zero.tidbcloud.com/) は、サインアップ不要でTiDB Cloudクラスタを作れるサービスです。
作成後 30 日間有効で、必要なら claim URL から TiDB Cloud Starter に移行できます。


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-quickstart")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 3. TiDB に接続してテーブルを作成する

`TableModel` を継承すると Python のクラスがそのまま TiDB のテーブル定義になります。
今回は3つのカラムのシンプルな `tasks` テーブルを作ります。

テーブルの作成は、`TiDBClient.create_table()` を呼び出します。繰り返し実行できるように、`if_exists="overwrite"` で作り直しています。


In [ ]:
from pytidb import TiDBClient
from pytidb.schema import Field, TableModel

DATABASE = "quickstart_demo"

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database=DATABASE,
    ensure_db=True,  # DB が無ければ作る
)
print("接続OK:", DATABASE)


class Task(TableModel):
    __tablename__ = "tasks"
    __table_args__ = {"extend_existing": True}

    id: int = Field(primary_key=True)
    title: str = Field()
    done: bool = Field(default=False)

# if_exists = "overwrite" でテーブルを作り直す
# if_exists = "skip" で既存のテーブルがあればそのまま使うこともできます
table = db.create_table(schema=Task, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


## 4. データを投入する (Create)

データのINSERTは、モデルクラスのインスタンスを作って `table.insert()` を呼び出すだけです。

単発の `insert` と、一括インサート用の `bulk_insert` の両方を試します。


In [ ]:
table.insert(Task(id=1, title="pytidb の README を読む"))
table.bulk_insert([
    Task(id=2, title="クイックスタートを試す"),
    Task(id=3, title="ベクトル検索を動かす"),
    Task(id=4, title="ハイブリッド検索を動かす"),
])
print(f"投入完了。現在の件数: {table.rows()}")


## 5. 取得する (Read)

`table.query()` は条件指定・件数制限・並び替えに対応します。
`.to_pydantic()` で `Task` クラスのインスタンスのリストが返り、カラムへのアクセスがPythonの属性アクセスになります。


In [ ]:
tasks = table.query(limit=10).to_pydantic()
for t in tasks:
    mark = "[x]" if t.done else "[ ]"
    print(f"{mark} #{t.id} {t.title}")


## 6. 更新と削除 (Update / Delete)

`update` は条件に一致する行を書き換え、`delete` は行を消します。


In [ ]:
# id=1 を完了扱いにする
table.update(values={"done": True}, filters={"id": 1})
# id=4 は削除する
table.delete(filters={"id": 4})

# 確認
for t in table.query(limit=10).to_pydantic():
    mark = "[x]" if t.done else "[ ]"
    print(f"{mark} #{t.id} {t.title}")
print(f"合計: {table.rows()} 件")


## 次のステップ

- `01_schema_and_types.ipynb` : TEXT / JSON / VECTOR などのデータ型の使い分け
- `02_query_and_filter.ipynb` : 複雑なフィルタ・並び替え・トランザクション
- `03_vector_search.ipynb` : 意味類似検索 (セマンティック検索)

## チャレンジ

- `Task` に `due_date`(日付)列を追加してみる (`datetime.date` 型)
- `table.query(filters={"done": False})` で未完了だけ取り出す
- `db.drop_table("tasks")` でテーブルを削除し、再実行する
